### Part (a)

In [1]:
from pathlib import Path
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import KFold, train_test_split
from IPython.display import display


In [2]:
np.random.seed(42)
folder = Path('.')
data = pd.read_csv(folder / 'retail.csv', dtype={'StockCode': str, 'CustomerID': str})
columns = ['StockCode', 'CustomerID']
display(pd.DataFrame({"rows": [len(data)], "columns": [data.shape[1]]}))
display(data[columns].nunique().rename("categories").to_frame())
display(data["Quantity"].describe().to_frame())


,rows,columns
0,397924,3


,categories
StockCode,3665
CustomerID,4339


,Quantity
count,397924.000000
mean,13.021823
std,180.420210
min,1.000000
25%,2.000000
50%,6.000000
75%,12.000000
max,80995.000000


In [3]:
(train, holdout) = train_test_split(np.arange(len(data)), test_size=0.3, random_state=42)
(val, test) = train_test_split(holdout, test_size=0.5, random_state=42)
pd.DataFrame({"split": ["train", "validation", "test"], "rows": [len(train), len(val), len(test)]})


,split,rows
0,train,278546
1,validation,59689
2,test,59689


### Part (b)

In [4]:
def target_encoding(data, train, holdout):
    folds = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = [(train[fit], train[fold]) for (fit, fold) in folds.split(train)]
    splits.append((train, holdout))
    encoded = np.empty((len(data), 2))
    for (fit, rows) in splits:
        fit_data = data.iloc[fit]
        for (j, column) in enumerate(['StockCode', 'CustomerID']):
            means = fit_data.groupby(column)['Quantity'].mean()
            encoded[rows, j] = data.iloc[rows][column].map(means).fillna(fit_data['Quantity'].mean()).to_numpy()
    return encoded


In [5]:
categories = []
encoded = target_encoding(data, train, holdout)
sizes = []
for column in columns:
    values = data.iloc[train][column].unique()
    mapping = {value: i + 1 for (i, value) in enumerate(values)}
    categories.append(data[column].map(mapping).fillna(0).to_numpy(dtype=np.int64))
    sizes.append(len(values))
categories = torch.tensor(np.column_stack(categories), dtype=torch.long)
encoded = np.log(encoded)
encoded = (encoded - encoded[train].mean(axis=0)) / encoded[train].std(axis=0)
encoded = torch.tensor(encoded, dtype=torch.float32)
y = data['Quantity'].to_numpy(dtype=np.float64)
target = torch.tensor(np.log1p(y), dtype=torch.float32)


In [6]:
class RetailModel(nn.Module):

    def __init__(self, method, sizes):
        super().__init__()
        self.method = method
        dimension = 0
        if method != 'Target encoding':
            self.stock = nn.Embedding(sizes[0] + 1, 16, padding_idx=0)
            self.customer = nn.Embedding(sizes[1] + 1, 16, padding_idx=0)
            dimension += 32
        if method != 'Embedding':
            dimension += 2
        self.net = nn.Sequential(nn.Linear(dimension, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, categories, encoded):
        inputs = []
        if self.method != 'Target encoding':
            inputs.extend([self.stock(categories[:, 0]), self.customer(categories[:, 1])])
        if self.method != 'Embedding':
            inputs.append(encoded)
        return self.net(torch.cat(inputs, dim=1)).squeeze(1)


In [7]:
def fit_retail(method):
    history = []
    torch.manual_seed(42)
    model = RetailModel(method, sizes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    best_loss = float('inf')
    wait = 0
    for epoch in range(50):
        model.train()
        order = torch.tensor(train)[torch.randperm(len(train))]
        for start in range(0, len(train), 1024):
            batch = order[start:start + 1024]
            optimizer.zero_grad()
            output = model(categories[batch], encoded[batch])
            loss = ((output - target[batch]) ** 2).mean()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            output = model(categories[val], encoded[val])
            val_loss = ((output - target[val]) ** 2).mean().item()
        history.append({'model': method, 'epoch': epoch + 1, 'val_log_mse': val_loss})
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1
        if wait == 5:
            break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        prediction_log = model(categories[test], encoded[test]).numpy().astype(np.float64)
    prediction = np.expm1(prediction_log)
    record = {'model': method, 'best_epoch': best_epoch, 'val_log_mse': best_loss, 'test_log_mse': np.mean((prediction_log - np.log1p(y[test])) ** 2), 'test_mae': np.mean(np.abs(prediction - y[test])), 'test_rmse': np.sqrt(np.mean((prediction - y[test]) ** 2))}
    return (model, record, history)


In [8]:
te_model, te_result, te_history = fit_retail("Target encoding")
pd.DataFrame([te_result])


,model,best_epoch,val_log_mse,test_log_mse,test_mae,test_rmse
0,Target encoding,22,0.387451,0.38838,6.927856,32.806744


### Part (c)

In [9]:
embedding_model, embedding_result, embedding_history = fit_retail("Embedding")
pd.DataFrame([embedding_result])


,model,best_epoch,val_log_mse,test_log_mse,test_mae,test_rmse
0,Embedding,29,0.29253,0.291603,6.223117,31.49066


### Part (d)

In [10]:
hybrid_model, hybrid_result, hybrid_history = fit_retail("Hybrid")
results = pd.DataFrame([te_result, embedding_result, hybrid_result])
results


,model,best_epoch,val_log_mse,test_log_mse,test_mae,test_rmse
0,Target encoding,22,0.387451,0.388380,6.927856,32.806744
1,Embedding,29,0.292530,0.291603,6.223117,31.490660
2,Hybrid,22,0.295415,0.294542,6.231761,31.608148


### Part (e)

In [11]:
histories = te_history + embedding_history + hybrid_history
results.to_csv(folder / 'results.csv', index=False)
pd.DataFrame(histories).to_csv(folder / 'training.csv', index=False)
pd.DataFrame([{'seed': 42, 'train_n': len(train), 'val_n': len(val), 'test_n': len(test), 'stock_categories': data.StockCode.nunique(), 'customer_categories': data.CustomerID.nunique(), 'train_stock_categories': sizes[0], 'train_customer_categories': sizes[1], 'val_unknown_stock': int((categories[val, 0] == 0).sum()), 'val_unknown_customer': int((categories[val, 1] == 0).sum()), 'test_unknown_stock': int((categories[test, 0] == 0).sum()), 'test_unknown_customer': int((categories[test, 1] == 0).sum())}]).to_csv(folder / 'run_details.csv', index=False)
